In [1]:
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import torch

# Gemma2_9b, ignores offshoreleaks (duecontext limit)

In [2]:
df = pd.read_excel("Results/BenchmarkDataset2025/finetuned_gemma2_9b_local/finetuned_gemma2_9b_local.xlsx")

In [3]:
len(df)

2128

In [5]:
empty_cypher_rows = df[df['generated_cypher'].isna() | (df['generated_cypher'] == '')]
for i in (empty_cypher_rows['database_name']):
    print(i)

offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
neoflix
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleaks
offshoreleak

In [2]:
def evaluate_model (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Results folder.
    Return the dataframe
    '''
    torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Results/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}.xlsx")

    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        df_results['cypher_errors_history'] = None           # Use None so we can store a list
        df_results['cypher_errors'] = None           # Use None so we can store a list
        df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list

    config = {'configurable': {'model': model, 'model_name':model_name, 'attempts_w_filtered':5, 'total_attempts':10},'recursion_limit':50}
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # results=[]
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
    # i=1
    # row = df_results.loc[i]
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i,question)
            # print(row['cypher'])
            
            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
            if database_name == 'offshoreleaks':
                print(f'Skipping row {i}, to avoid memory error in {database_name}')
                continue
            else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                torch.cuda.empty_cache() #clear gpu cache
            
                # Run the graph and save outputs
                result =app.invoke(inputs, config =config) #,debug=False
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                # df_results.at[i, 'path'] = result['path']
                # df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                # df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                # df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                # df_results.at[i, 'answer'] = result['answer']
                # if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                #     df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                # else:
                #     df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                # results.append(result)
                # torch.cuda.empty_cache()#clear gpu cache
                
                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
        else:
            print(f"Skipping row {i}, already computed.")
            
    return df_results

In [3]:
from Configuration.models import models #import models to evaluate
model_name = 'finetuned_gemma2_9b_local'
df_results = evaluate_model (model_name, models[model_name](), 'benchmark2025')

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


Starting evaluation for finetuned_gemma2_9b_local on benchmark2025 ...


  0%|          | 0/2128 [00:00<?, ?it/s]

0 Identify the 5 suppliers with the highest average unit price of products supplied.
Cypher Generator: START
1 List the first 3 genres of movies with the lowest average revenue.
Cypher Generator: START
2 What is the total number of users who are VIPs for at least three different streams and are also moderators for at least one stream?
Cypher Generator: START
3 "show me the oldest tweet id"
Cypher Generator: START
4 List the 5 most recent users who started following 'Neo4j'.
Cypher Generator: START
5 Which users are amplified by 'Me' according to the AMPLIFIES relationship?
Cypher Generator: START
6 Who are the top 5 users that a specific user named 'Neo4j' follows?
Cypher Generator: START
7 What is the average number of followers for users who are both chatters and VIPs of the same stream?
Cypher Generator: START
8 List 3 questions with the lowest score.
Cypher Generator: START
9 Find the `customerID` and `shipCity` for the customer with the highest total order value (sum of `unitPrice

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


10 List entities that have undergone a company type change from the original registration.
Skipping row 10, to avoid memory error in offshoreleaks
11 What is the average `unitPrice` of products that have been ordered in quantities greater than 10?
Cypher Generator: START
12 Which users have an area less than 3 and have interacted with 'inchristbl.bsky.social'?
Cypher Generator: START
13 Find the entities with the country_codes 'HKG' and service provider 'Mossack Fonseca'.
Skipping row 13, to avoid memory error in offshoreleaks
14 Display the first 3 users who have interacted with someone located at y values less than -5000.
Cypher Generator: START
15 Find the top 5 movies with the most votes.
Cypher Generator: START
16 Identify the top 3 users by the number of people they are following.
Cypher Generator: START
17 Who does 'neo4j' interact with most frequently?
Cypher Generator: START
18 List all movies with a 'Pretty funny at times' review summary.
Cypher Generator: START
19 Find the u

KeyboardInterrupt: 

In [ ]:
# from peft import PeftModel, PeftConfig
# import torch
# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer,
#     BitsAndBytesConfig,
# )


# instruction = (
#     "Generate Cypher statement to query a graph database. "
#     "Use only the provided relationship types and properties in the schema. \n"
#     "Schema: {schema} \n Question: {question}  \n Cypher output: "
# )





# def _postprocess_output_cypher(output_cypher: str) -> str:
#     # Remove any explanation. E.g.  MATCH...\n\n**Explanation:**\n\n -> MATCH...
#     # Remove cypher indicator. E.g.```cypher\nMATCH...```` --> MATCH...
#     # Note: Possible to have both:
#     #   E.g. ```cypher\nMATCH...````\n\n**Explanation:**\n\n --> MATCH...
#     partition_by = "**Explanation:**"
#     output_cypher, _, _ = output_cypher.partition(partition_by)
#     output_cypher = output_cypher.strip("`\n")
#     output_cypher = output_cypher.lstrip("cypher\n")
#     output_cypher = output_cypher.strip("`\n ")
#     return output_cypher

# # Model
# model_name = "neo4j/text2cypher-gemma-2-9b-it-finetuned-2024v1"
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     torch_dtype=torch.bfloat16,
#     attn_implementation="eager",
#     low_cpu_mem_usage=True,
# )

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

TemplateError: Conversation roles must alternate user/assistant/user/assistant/...

In [ ]:
# def prepare_chat_prompt(question, schema) -> list[dict]:
#     chat = [
#         {
#             "role": "user",
#             "content": instruction.format(
#                 schema=schema, question=question
#             ),
#         }
#     ]
#     return chat
# # Question
# question = "What are the movies of Tom Hanks?"
# schema = "(:Actor)-[:ActedIn]->(:Movie)" # Check the NOTE below on creating your own schemas
# new_message = prepare_chat_prompt(question=question, schema=schema)
# prompt = tokenizer.apply_chat_template(new_message, add_generation_prompt=True, tokenize=False)
# inputs = tokenizer(prompt, return_tensors="pt", padding=True)

# # Any other parameters
# model_generate_parameters = {
#     "top_p": 0.9,
#     "temperature": 0.2,
#     "max_new_tokens": 512,
#     "do_sample": True,
#     "pad_token_id": tokenizer.eos_token_id,
# }

# inputs.to(model.device)
# model.eval()
# with torch.no_grad():
#     tokens = model.generate(**inputs, **model_generate_parameters)
#     tokens = tokens[:, inputs.input_ids.shape[1] :]
#     raw_outputs = tokenizer.batch_decode(tokens, skip_special_tokens=True)
#     outputs = [_postprocess_output_cypher(output) for output in raw_outputs]
    
# print(outputs)
# # > ["MATCH (a:Actor {Name: 'Tom Hanks'})-[:ActedIn]->(m:Movie) RETURN m"]


["MATCH (a:Actor {name: 'Tom Hanks'})-[:ActedIn]->(m:Movie) RETURN m"]


# TEST

In [ ]:
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.runnables.config import RunnableConfig
# from huggingface_hub import InferenceClient

# from Nodes.States.states import  OverallState
# # from Configuration.models import models

# from Configuration.models import models #import models to evaluate
# model_name = 'finetuned_gemma2_9b'

# system_prompt = """
#                 Task: Generate Cypher statement to query a graph database. 
#                 Instructions: Use only the provided relationship types and properties in the schema.
#                 Do not use any other relationship types or properties that are not provided in the schema.
#                 Do not include any explanations or apologies in your responses.
#                 Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
#                 Do not include any text except the generated Cypher statement.
#                 """
# human_prompt =  """
#                 Generate Cypher statement to query a graph database,as plain text without any code block formatting or backticks.
#                 Use only the provided relationship types and properties in the schema.
#                 \nSchema: {schema}
#                 \nQuestion: {question}
#                 \nCypher statement:
                
#                 """
# text2cypher_prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             (
#                 system_prompt
#             ),
#         ),
#         (   #as plain text without any code block formatting or backticks : ADDED
#             "human",
#             (
#               human_prompt
#             ),
#         ),
#     ]
# )

# text2cypher_prompt_flattened = ChatPromptTemplate.from_messages(
#     [
#         # (
#         #     "system",
#         #     (
#         #         system_prompt
#         #     ),
#         # ),
#         (   #as plain text without any code block formatting or backticks : ADDED
#             "human",
#             (
#               system_prompt + human_prompt
#             ),
#         ),
#     ]
# )

# chat_model = models[model_name]() 



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


OutOfMemoryError: CUDA out of memory. Tried to allocate 98.00 MiB. GPU 0 has a total capacity of 14.57 GiB of which 22.75 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.41 GiB is allocated by PyTorch, and 29.10 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# # model_name = config['configurable']['model_name']
# question = "How many are the actors?"
# schema = "(:Actor)-[:ActedIn]->(:Movie)" 

# if model_name == 'gemma2_9b' or model_name == 'finetuned_gemma2_9b':
#     text2cypher_chain = text2cypher_prompt_flattened | chat_model | StrOutputParser()
# else:    
#     text2cypher_chain = text2cypher_prompt | chat_model | StrOutputParser()

# generated_cypher = text2cypher_chain.invoke(
#     {
#         "question": question,
#         "schema": schema,
#     }
# )

# if model_name == 'llama3_2_3b_local':
#     generated_cypher = generated_cypher.split('assistant<|end_header_id|>')[1].strip()
# if model_name == 'finetuned_gemma2_9b':
#     generated_cypher = generated_cypher.split('<start_of_turn>model\n')[1].strip()
# print('Cypher Generator: START')
# generated_cypher

Cypher Generator: START


'MATCH (a:Actor) RETURN count(a)'